In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from peft import PeftModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading weights to {device}...")

# Load DeBERTa Base + LoRA Adapters
deb_base = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-small")
deberta_model = PeftModel.from_pretrained(deb_base, "/kaggle/input/models/tanmay240/deberta-v3-small-checkpoint/pytorch/default/1/deberta-v3-small_final").to(device).eval()
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

# Load RoBERTa Base + LoRA Adapters
rob_base = AutoModelForMultipleChoice.from_pretrained("roberta-base")
roberta_model = PeftModel.from_pretrained(rob_base, "/kaggle/input/models/tanmay240/roberta-base-checkpoint/pytorch/default/1/roberta-base_final").to(device).eval()
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Load Data
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

option_letters = ['A', 'B', 'C', 'D', 'E']
label_to_id = {letter: idx for idx, letter in enumerate(option_letters)}
id_to_label = {idx: letter for letter, idx in label_to_id.items()}

def get_probabilities(model, tokenizer, prompt, row):
    """Formats MCQ inputs, executes forward pass, and applies Softmax."""
    first_sentences = [prompt] * 5
    second_sentences = [str(row[letter]) for letter in option_letters]
    
    inputs = tokenizer(first_sentences, second_sentences, padding=True, truncation=True, max_length=128, return_tensors="pt")
    input_ids = inputs['input_ids'].unsqueeze(0).to(device)
    attention_mask = inputs['attention_mask'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze(0)
        probs = F.softmax(logits, dim=-1)
    return probs.cpu().numpy()

# Milestone-5
row_25 = test_df.iloc[25]
prompt_25 = str(row_25['prompt'])

p_deb_25 = get_probabilities(deberta_model, deberta_tokenizer, prompt_25, row_25)
p_rob_25 = get_probabilities(roberta_model, roberta_tokenizer, prompt_25, row_25)

# Q1
best_deb_idx = np.argmax(p_deb_25)
print(f"Q1: {id_to_label[best_deb_idx]}, {p_deb_25[best_deb_idx]:.4f}")

# Q2
p_avg_25 = (p_deb_25 + p_rob_25) / 2
print(f"Q2: {id_to_label[np.argmax(p_avg_25)]}")

# Q3
p_weight_25 = (0.70 * p_deb_25) + (0.30 * p_rob_25)
print(f"Q3: {id_to_label[np.argmax(p_weight_25)]}")

# Q4
top_3_indices = np.argsort(p_weight_25)[::-1][:3]
top_3_str = " ".join([id_to_label[i] for i in top_3_indices])
print(f"Q4: {top_3_str}")

# Q5
submission_list = []
for idx, row in test_df.iterrows():
    p_deb = get_probabilities(deberta_model, deberta_tokenizer, str(row['prompt']), row)
    p_rob = get_probabilities(roberta_model, roberta_tokenizer, str(row['prompt']), row)
    p_weight = (0.70 * p_deb) + (0.30 * p_rob)
    top_3 = " ".join([id_to_label[i] for i in np.argsort(p_weight)[::-1][:3]])
    submission_list.append({'id': row['id'], 'Prediction': top_3})

pd.DataFrame(submission_list).to_csv('submission.csv', index=False)
print(f"Q5: {len(submission_list)}")

# Q6
tta_changes = 0
for idx in range(50):
    row = test_df.iloc[idx]
    orig_prompt = str(row['prompt'])
    tta_prompt = "Answer the following multiple-choice question carefully: " + orig_prompt
    
    p_orig = get_probabilities(deberta_model, deberta_tokenizer, orig_prompt, row)
    p_tta = get_probabilities(deberta_model, deberta_tokenizer, tta_prompt, row)
    p_avg = (p_orig + p_tta) / 2
    
    if np.argmax(p_orig) != np.argmax(p_avg):
        tta_changes += 1
print(f"Q6: {tta_changes}")

# Ensemble Metrics
diff_top1 = 0
positive_gain = 0
top3_changes = 0

for idx in range(100):
    row = test_df.iloc[idx]
    prompt = str(row['prompt'])
    
    p_deb = get_probabilities(deberta_model, deberta_tokenizer, prompt, row)
    p_rob = get_probabilities(roberta_model, roberta_tokenizer, prompt, row)
    p_ens = (0.70 * p_deb) + (0.30 * p_rob)
    
    # Q7
    if np.argmax(p_deb) != np.argmax(p_ens):
        diff_top1 += 1
        
    # Q8
    if np.max(p_ens) > np.max(p_deb):
        positive_gain += 1
        
    # Q9
    t3_deb = np.argsort(p_deb)[::-1][:3]
    t3_ens = np.argsort(p_ens)[::-1][:3]
    if not np.array_equal(t3_deb, t3_ens):
        top3_changes += 1

print(f"Q7: {diff_top1}")
print(f"Q8: {positive_gain}")
print(f"Q9: {top3_changes}")

# Q10
def map_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1.0)
    return 0.0

map_scores = []
for idx in range(100):
    row = train_df.iloc[idx]
    prompt = str(row['prompt'])
    actual_ans = row['answer']
    
    p_deb = get_probabilities(deberta_model, deberta_tokenizer, prompt, row)
    p_rob = get_probabilities(roberta_model, roberta_tokenizer, prompt, row)
    p_ens = (0.70 * p_deb) + (0.30 * p_rob)
    
    top_3_preds = [id_to_label[i] for i in np.argsort(p_ens)[::-1][:3]]
    map_scores.append(map_at_3(actual_ans, top_3_preds))

print(f"Q10: {np.mean(map_scores):.4f}")

Loading weights to cuda...


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q1: D, 0.2086
Q2: D
Q3: D
Q4: D C E
Q5: 500
Q6: 15
Q7: 3
Q8: 0
Q9: 7
Q10: 0.4333
